# Experiment 5: Semantic Search Engine Using Sentence Embeddings and FAISS

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To build a semantic document search engine using SentenceTransformers to generate embeddings and FAISS for nearest-neighbour search, and to compare it with traditional keyword (TF-IDF) search on retrieval quality metrics.

## 2. Objectives

- To understand the difference between keyword search and semantic search.
- To create a TF-IDF keyword search baseline with scikit-learn.
- To convert documents and queries into sentence embeddings with SentenceTransformers.
- To build exact and approximate FAISS indexes and search them.
- To evaluate the search engines using Recall@10, MRR@10 and nDCG@10.
- To compare the speed and accuracy of the exact and approximate indexes.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- Internet connection for downloading the model and the dataset
- Required Python libraries: `scikit-learn`, `sentence-transformers`, `faiss-cpu`, `datasets`, `pandas`, `numpy`

## 4. Theory

### A. Keyword search with TF-IDF

TF-IDF represents a document as a sparse vector over the vocabulary. A word gets a high weight if it is frequent in the document (term frequency) but rare in the collection (inverse document frequency). A query is scored against each document by cosine similarity. It works only when the query and the document share the same words.

### B. Semantic search with sentence embeddings

A sentence-embedding model (here `all-MiniLM-L6-v2`) maps a text to a dense vector of 384 numbers. Texts with similar meaning get nearby vectors even when they use different words, for example "heart attack" and "myocardial infarction". Search becomes a nearest-neighbour problem in vector space.

### C. FAISS

FAISS is a library for fast vector search.

| Index | Type | Behaviour |
|---|---|---|
| `IndexFlatIP` | exact | Compares the query with every vector. With normalised vectors the inner product equals cosine similarity. |
| `IndexHNSWFlat` | approximate | Follows a graph of neighbours. Much faster on large collections, but may miss a few true neighbours. |

### D. Retrieval metrics

| Metric | Meaning |
|---|---|
| Recall@10 | Fraction of the relevant documents found in the top 10 results. |
| MRR@10 | Average of 1/rank of the first relevant result (0 if none in the top 10). |
| nDCG@10 | Rewards relevant documents that appear higher in the ranking. |

The benchmark is **SciFact**: 5,183 scientific abstracts and 300 test claims, each with the abstract(s) that support or refute it.

## 5. Algorithm / Procedure

1. Install and import the libraries.
2. Load the SciFact corpus, queries and relevance judgements.
3. Build the TF-IDF index and rank the documents for every query.
4. Encode the documents and queries with SentenceTransformers.
5. Build the FAISS exact index and the HNSW approximate index and search them.
6. Compute Recall@10, MRR@10 and nDCG@10 and the search time for every method.
7. Compare the results of one example query and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q scikit-learn sentence-transformers faiss-cpu datasets pandas numpy

### Step 2: Load the SciFact dataset

Each query is a scientific claim. `relevant[i]` is the set of document positions that are relevant to query `i`.

In [2]:
import time, warnings, numpy as np, pandas as pd
from datasets import load_dataset
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 180)

corpus = load_dataset("mteb/scifact", "corpus")["corpus"]
queries = load_dataset("mteb/scifact", "queries")["queries"]
qrels = load_dataset("mteb/scifact", "default")["test"]

pos = {d: i for i, d in enumerate(corpus["_id"])}
relevant = {}
for r in qrels:
    relevant.setdefault(r["query-id"], set()).add(pos[r["corpus-id"]])
qtext = dict(zip(queries["_id"], queries["text"]))
q_ids = [q for q in qtext if q in relevant]
q_texts, rel = [qtext[q] for q in q_ids], [relevant[q] for q in q_ids]
docs = [t + ". " + x for t, x in zip(corpus["title"], corpus["text"])]   # title + abstract
print(len(docs), "documents |", len(q_texts), "queries")
print("Example claim:", q_texts[3])

5183 documents | 300 queries
Example claim: 5% of perinatal mortality is due to low birth weight.


### Step 3: Evaluation function

`evaluate` takes the ranked document positions for every query and returns the three metrics averaged over all queries.

In [3]:
def evaluate(ranked):
    discount = 1 / np.log2(np.arange(2, 12))
    out = []
    for r, rl in zip(ranked, rel):
        hit = np.array([d in rl for d in r[:10]], dtype=float)
        first = np.argmax(hit) if hit.any() else None
        out.append([hit.sum() / len(rl), 0 if first is None else 1 / (first + 1),
                    (hit * discount).sum() / discount[:min(len(rl), 10)].sum()])
    return dict(zip(["Recall@10", "MRR@10", "nDCG@10"], [round(float(v), 3) for v in np.mean(out, axis=0)]))

### Step 4: Keyword search with TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
X = tfidf.fit_transform(docs)                 # sparse matrix: documents x vocabulary
print("TF-IDF matrix:", X.shape, "| vocabulary size:", len(tfidf.vocabulary_))

t0 = time.time()
scores = linear_kernel(tfidf.transform(q_texts), X)      # cosine similarity of every query with every document
tfidf_rank = np.argsort(-scores, axis=1)[:, :10]
tfidf_time = (time.time() - t0) / len(q_texts) * 1000
print("TF-IDF:", evaluate(tfidf_rank))

TF-IDF matrix: (5183, 35617) | vocabulary size: 35617
TF-IDF: {'Recall@10': 0.774, 'MRR@10': 0.589, 'nDCG@10': 0.628}


### Step 5: Encode the documents and queries with SentenceTransformers

`normalize_embeddings=True` makes every vector length 1, so the inner product equals the cosine similarity. Encoding the 5,183 documents is the slow step; it is done once.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_emb = model.encode(docs, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
q_emb = model.encode(q_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print("Document embeddings:", doc_emb.shape, "| query embeddings:", q_emb.shape)

Document embeddings: (5183, 384) | query embeddings: (300, 384)


### Step 6: Build the FAISS indexes and search

The exact index compares the query with all documents. The HNSW index is an approximate graph index; `M=32` is the number of neighbours per node and `efSearch` controls how many nodes are visited at query time.

In [6]:
import faiss

d = doc_emb.shape[1]
flat = faiss.IndexFlatIP(d)                   # exact search with inner product
flat.add(doc_emb)

hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efSearch = 64
hnsw.add(doc_emb)
print("Vectors in the exact index:", flat.ntotal, "| in the HNSW index:", hnsw.ntotal)

Vectors in the exact index: 5183 | in the HNSW index: 5183


In [7]:
def search(index):
    t0 = time.time()
    _, ids = index.search(q_emb, 10)
    return ids, (time.time() - t0) / len(q_emb) * 1000

flat_rank, flat_time = search(flat)
hnsw_rank, hnsw_time = search(hnsw)
print("FAISS exact:", evaluate(flat_rank))
print("FAISS HNSW :", evaluate(hnsw_rank))

FAISS exact: {'Recall@10': 0.788, 'MRR@10': 0.607, 'nDCG@10': 0.648}


FAISS HNSW : {'Recall@10': 0.788, 'MRR@10': 0.607, 'nDCG@10': 0.648}


### Step 7: Compare a query side by side

In [8]:
# pick the first query where the semantic top result is relevant but TF-IDF finds nothing in its top 10
i = next(j for j in range(len(q_texts)) if flat_rank[j][0] in rel[j] and not set(tfidf_rank[j]) & rel[j])
print("Claim:", q_texts[i], "\nRelevant document(s):", [corpus["title"][j][:70] for j in rel[i]], "\n")
pd.DataFrame({"TF-IDF": [corpus["title"][j][:60] + (" *" if j in rel[i] else "") for j in tfidf_rank[i][:4]],
              "FAISS (MiniLM)": [corpus["title"][j][:60] + (" *" if j in rel[i] else "") for j in flat_rank[i][:4]]},
             index=[1, 2, 3, 4])

Claim: Breast cancer development is determined exclusively by genetic factors. 
Relevant document(s): ['Gene–environment interactions in 7610 women with breast cancer: prospe'] 



,TF-IDF,FAISS (MiniLM)
1,Risk for breast cancer development determined by mammographi,Gene–environment interactions in 7610 women with breast canc *
2,Management of women who have a genetic predisposition for br,Aberrant luminal progenitors as the candidate target populat
3,"Diabetes, metabolic syndrome, and breast cancer: a review of",A common variant in BRCA2 is associated with both breast can
4,Breast cancer and obesity: an update.,Heterogenic Loss of the Wild-Type BRCA Allele in Human Breas


## 7. Experimental Observations

### Observation Table 1: Retrieval quality and speed

In [9]:
overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(flat_rank, hnsw_rank)])
pd.DataFrame({"TF-IDF (keyword)": {**evaluate(tfidf_rank), "ms per query": round(tfidf_time, 2)},
              "FAISS exact (MiniLM)": {**evaluate(flat_rank), "ms per query": round(flat_time, 2)},
              "FAISS HNSW (MiniLM)": {**evaluate(hnsw_rank), "ms per query": round(hnsw_time, 2)}}).T

,Recall@10,MRR@10,nDCG@10,ms per query
TF-IDF (keyword),0.774,0.589,0.628,0.20
FAISS exact (MiniLM),0.788,0.607,0.648,0.52
FAISS HNSW (MiniLM),0.788,0.607,0.648,0.04


### Observation Table 2: Effect of `efSearch` on the HNSW index

In [10]:
rows = []
for ef in [4, 16, 64, 256]:
    hnsw.hnsw.efSearch = ef
    ids, ms = search(hnsw)
    rows.append({"efSearch": ef, "Overlap with exact top-10": round(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(flat_rank, ids)]), 3),
                 "nDCG@10": evaluate(ids)["nDCG@10"], "ms per query": round(ms, 3)})
pd.DataFrame(rows).set_index("efSearch")

,Overlap with exact top-10,nDCG@10,ms per query
efSearch,,,
4,0.861,0.596,0.008
16,0.966,0.628,0.017
64,0.996,0.648,0.052
256,1.000,0.648,0.202


**Observations**

- The TF-IDF baseline (with stop words removed and sublinear term frequency) is already strong on this scientific text: Recall@10 0.774, MRR@10 0.589, nDCG@10 0.628. Scientific claims share many technical terms with their abstracts, which favours keyword matching.
- Semantic search with MiniLM embeddings and an exact FAISS index is better on every metric: Recall@10 0.788, MRR@10 0.607, nDCG@10 0.648. The gain is real but modest (about 2 points of nDCG@10) for a small general-purpose model.
- The example claim ("Breast cancer development is determined exclusively by genetic factors") shows why. TF-IDF returned papers that repeat the words "breast cancer", but not the relevant study. The embedding model ranked the relevant paper on gene-environment interactions first, and its other results (BRCA variants, genetic alleles) are about the same topic.
- The HNSW index returned the same quality as the exact index (identical metrics) and was about thirteen times faster per query (0.04 ms against 0.52 ms). The speed difference grows with the collection size.
- `efSearch` controls the accuracy-speed trade-off of HNSW: with 4 the top-10 overlaps with the exact result by only 86 percent, with 64 by 99.6 percent, and with 256 it is identical, while the time per query rises from 0.01 ms to 0.20 ms.

## 8. Result

A semantic search engine was built with SentenceTransformers (all-MiniLM-L6-v2) and FAISS on the SciFact collection of 5,183 abstracts and 300 queries, and compared with TF-IDF keyword search. Semantic search with the exact FAISS index achieved Recall@10 0.788, MRR@10 0.607 and nDCG@10 0.648, against 0.774, 0.589 and 0.628 for TF-IDF. The approximate HNSW index gave the same quality at about thirteen times the speed of the exact index. Increasing efSearch moved the HNSW results from 86 percent to 100 percent agreement with the exact search.

## 9. Conclusion

Semantic search using sentence embeddings retrieves documents by meaning, so it can find relevant text that shares few words with the query, while TF-IDF depends on exact word overlap. FAISS makes the nearest-neighbour search fast: the exact index is the most accurate, and the HNSW index trades a small loss of accuracy for speed that becomes important as the collection grows to millions of documents. The best choice depends on the data, and combining keyword and semantic scores is often used in practice.